<a href="https://colab.research.google.com/github/PanthegrammerPRO/Diplwmatikh/blob/main/Running_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Σύνδεση με το Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Εγκατάσταση εξαρτημάτων
# Εγκαθιστούμε το unsloth με --no-deps (όπως προτείνει το repo τους για Colab)
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

!pip install --no-deps xformers trl peft accelerate bitsandbytes unsloth_zoo

# 3. Refresh το περιβάλλον της Python
import site
from importlib import reload
reload(site)

print("\nΗ προετοιμασία ολοκληρώθηκε! Το περιβάλλον είναι έτοιμο.")

Mounted at /content/drive
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-svpnm5i3/unsloth_4a1f87b3b6024357ac8042f13694ee5b
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-svpnm5i3/unsloth_4a1f87b3b6024357ac8042f13694ee5b
  Resolved https://github.com/unslothai/unsloth.git to commit 9bab3b955ec3be719178aa43cd69a51ff479d0af
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.5.1-py3-none-any.whl size=32273190 sha256=1948973543408e30f66d96025413c73bc31862cbd46807edcacc8c937aafcb77
  Stored in directory: /tmp/pip-ephem-wheel-cache-03nbu5cj/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 60.7 

In [ ]:
import unsloth
from unsloth import FastLanguageModel
import torch

# 1. Μονοπάτι στο Drive
model_path = "/content/drive/MyDrive/Llama3_files"

# 2. Φόρτωση Μοντέλου & Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 2048,
    load_in_4bit = True,   # Απαραίτητο για τη μνήμη του Colab
    dtype = None,          # Αυτόματη επιλογή format
)

tokenizer.pad_token = tokenizer.eos_token # Ορισμός pad token
tokenizer.padding_side = "left"           # Padding στα αριστερά για σωστό generation

# 3. Ενεργοποίηση Inference
FastLanguageModel.for_inference(model)

# 4. Έλεγχος Μνήμης (Απλοποιημένος)
gpu_stats = torch.cuda.get_device_properties(0)
reserved_gb = torch.cuda.memory_reserved(0) / 1024 / 1024 / 1024
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 2)

print(f"\nΤο Llama 3 είναι έτοιμο!")
print(f"VRAM σε χρήση: {reserved_gb:.2f} / {max_memory} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.1: Fast Llama patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /content/drive/MyDrive/Llama3_files as a legacy tokenizer.


/content/drive/MyDrive/Llama3_files does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.

Το Llama 3 είναι έτοιμο!
VRAM σε χρήση: 5.40 / 14.56 GB


In [ ]:
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import re
from sklearn.metrics import classification_report

# Προετοιμασία και καθαρισμός test dataset
print("Φόρτωση και καθαρισμός δεδομένων ελέγχου (Test Set)...")
label_names = ["sadness", "joy", "love", "anger", "fear", "surprise"]

# Φορτώνουμε το test set απευθείας από το Hugging Face
raw_test = load_dataset("dair-ai/emotion", split="test")
df_test = pd.DataFrame(raw_test)

# Καθαρίζουμε το test set από εσωτερικές διπλοεγγραφές
initial_len = len(df_test)
df_test_cleaned = df_test.drop_duplicates(subset=['text']).copy()
print(f"Test Set: Αφαιρέθηκαν {initial_len - len(df_test_cleaned)} διπλότυπα. Νέο μέγεθος: {len(df_test_cleaned)}")

# Το μετατρέπουμε πάλι σε αντικείμενο Hugging Face Dataset
clean_test_dataset = Dataset.from_pandas(df_test_cleaned)

# Επιλογή δειγμάτων με seed
test_samples = clean_test_dataset.shuffle(seed=69).select(range(1000))

# Ορισμός Τεχνικών Prompting
def get_prompt(text, choice):
    options = ", ".join(label_names)

    if choice == "1":  # Zero-Shot
        return (
            f"Classify the emotion in the following text.\n"
            f"You must respond with exactly one word from this list: [{options}]. Do not add any explanation.\n\n"
            f"Text: '{text}'\n"
            f"Emotion:"
        )

    elif choice == "2":  # Role-Prompting
        return (
            f"You are a specialized NLP sentiment analyst with deep expertise in fine-grained emotion classification.\n"
            f"Your task is to identify the single most dominant emotion expressed in the text.\n"
            f"Respond with exactly one word from this list: [{options}]. Output the label only — no explanation.\n\n"
            f"Text: '{text}'\n"
            f"Emotion label:"
        )

    elif choice == "3":  # Few-Shot
        return (
            f"Classify the emotion of the text. Respond with exactly one word from: [{options}].\n\n"
            f"Text: 'I feel like I am actually making a difference in this world'\n"
            f"Emotion: joy\n\n"
            f"Text: 'I feel so discouraged and like I am failing everyone'\n"
            f"Emotion: sadness\n\n"
            f"Text: 'I am so done with people treating me this way'\n"
            f"Emotion: anger\n\n"
            f"Text: 'I cannot stop thinking about how much I care for them'\n"
            f"Emotion: love\n\n"
            f"Text: 'I am feeling a bit apprehensive about the upcoming changes'\n"
            f"Emotion: fear\n\n"
            f"Text: 'I never thought this day would come, I am speechless'\n"
            f"Emotion: surprise\n\n"
            f"Text: 'I just feel empty and like nothing will ever get better'\n"
            f"Emotion: sadness\n\n"
            f"Text: '{text}'\n"
            f"Emotion:"
        )

    elif choice == "4":  # Chain of Thought
        return (
            f"Classify the emotion of the text. Choose strictly one label from: [{options}].\n\n"
            f"Follow these two steps:\n"
            f"Step 1: In one sentence, identify the key emotional words or phrases in the text.\n"
            f"Step 2: Output exactly 'Final Label: <emotion>' using one word from the list above.\n\n"
            f"Text: '{text}'\n"
            f"Step 1:"
        )

    elif choice == "5":  # Few-Shot Chain of Thought
        return (
            f"You are an expert NLP analyst. Classify the emotion of the text into exactly one of the following labels: [{options}].\n\n"
            f"Definitions:\n"
            f"- joy: Happiness, excitement, relief.\n"
            f"- sadness: Grief, feeling down, disappointment.\n"
            f"- anger: Frustration, irritation, outrage.\n"
            f"- fear: Anxiety, nervousness, terror.\n"
            f"- love: Affection, adoration, care.\n"
            f"- surprise: Astonishment, unexpected events.\n\n"
            f"Analyze the text by thinking step-by-step. First identify the trigger, then output the final label.\n\n"
            f"Text: 'I am feeling a bit apprehensive about the upcoming changes'\n"
            f"Analysis: The trigger is 'upcoming changes'. The word 'apprehensive' strongly indicates nervousness about the future. This aligns with fear.\n"
            f"Final Label: fear\n\n"
            f"Text: 'I feel such a strong affection for someone special in my life'\n"
            f"Analysis: The trigger is 'someone special'. 'Strong affection' indicates a deep personal bond, which aligns with love rather than just joy.\n"
            f"Final Label: love\n\n"
            f"Now analyze the following text:\n"
            f"Text: '{text}'\n"
            f"Analysis:"
        )

    elif choice == "6":  # Emotion Prompting
        return (
            f"I really need your help with this — it is important to me that you get it right.\n"
            f"Please read the text carefully and identify the emotion it expresses.\n"
            f"Respond with exactly one word from this list: [{options}]. No explanation needed.\n\n"
            f"Text: '{text}'\n"
            f"Emotion:"
        )

    return None

# Μενού και εκτέλεση
print("\n--- LLM Emotion Classification Tester ---")
print("1: Zero-Shot\n2: Role Prompting\n3: Few-Shot\n4: Chain-of-thought\n5: Few-Shot CoT\n6: Emotion Prompting")
choice = input("\nEnter choice (1,2,3,4,5,6): ").strip()

mode_name = {"1": "Zero-Shot", "2": "Role Prompting", "3": "Few-Shot", "4": "Chain-of-thought", "5": "Few-Shot CoT", "6": "Emotion Prompting"}.get(choice, "Unknown")

y_true = []
y_pred = []

print(f"\nRunning test using {mode_name}...")

# Ρυθμίσεις Batching
batch_size = 32
current_max_tokens = 60 if choice in ["4", "5"] else 5

# Loop (Με χρήση Batching)
for i in tqdm(range(0, len(test_samples), batch_size)):

    # Επιλογή των δειγμάτων για το τρέχον batch
    batch_items = test_samples.select(range(i, min(i + batch_size, len(test_samples))))

    prompts = [get_prompt(item['text'], choice) for item in batch_items]
    actual_labels = [label_names[item['label']] for item in batch_items]

    # Tokenization ολόκληρου του batch
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    input_length = inputs.input_ids.shape[1]

    # Παραγωγή κειμένου για όλο το batch ταυτόχρονα
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=current_max_tokens,
            temperature=0.01,
            pad_token_id=tokenizer.eos_token_id
        )

    # Αποκωδικοποίηση και επεξεργασία της κάθε απάντησης ξεχωριστά
    for j, output in enumerate(outputs):
        generated_tokens = output[input_length:]
        prediction_raw = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip().lower()

        # Καθαρισμός και Εξαγωγή Label
        if choice in ["4", "5"]:
            prediction = ""
            match = re.search(r'final label:\s*([a-z]+)', prediction_raw)
            if match:
                prediction = match.group(1).strip()
            else:
                found_labels = [label for label in label_names if label in prediction_raw]
                prediction = found_labels[-1] if found_labels else "unknown"
        else:
            words = re.findall(r'\w+', prediction_raw)
            prediction = words[0] if words else "unknown"

        y_true.append(actual_labels[j])
        y_pred.append(prediction)


# Τελικά Αποτελέσματα
print(f"\n{'='*40}")
print(f" FINAL RESULTS: {mode_name}")
print(f"{'='*40}")

# Υπολογισμός Accuracy
correct_predictions = sum(1 for true, pred in zip(y_true, y_pred) if true == pred)
accuracy = (correct_predictions / len(y_true)) * 100
print(f"Overall Accuracy: {accuracy:.2f}%\n")

# Εκτύπωση Classification Report
print("--- Classification Report ---")
print(classification_report(y_true, y_pred, labels=label_names, zero_division=0))

Φόρτωση και καθαρισμός δεδομένων ελέγχου (Test Set)...


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Test Set: Αφαιρέθηκαν 0 διπλότυπα. Νέο μέγεθος: 2000

--- LLM Emotion Classification Tester ---
1: Zero-Shot
2: Role Prompting
3: Few-Shot
4: Chain-of-thought
5: Few-Shot CoT
6: Emotion Prompting

Enter choice (1,2,3,4,5,6): 1

Running test using Zero-Shot...


100%|██████████| 32/32 [01:47<00:00,  3.37s/it]


 FINAL RESULTS: Zero-Shot
Overall Accuracy: 43.50%

--- Classification Report ---
              precision    recall  f1-score   support

     sadness       0.62      0.56      0.59       303
         joy       0.78      0.29      0.42       328
        love       0.19      0.69      0.30        89
       anger       0.52      0.42      0.46       136
        fear       0.69      0.32      0.44       114
    surprise       0.15      0.53      0.23        30

   micro avg       0.44      0.43      0.44      1000
   macro avg       0.49      0.47      0.41      1000
weighted avg       0.61      0.43      0.46      1000



In [ ]:
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import re
from sklearn.metrics import classification_report

# Προετοιμασία και καθαρισμός test dataset
print("Φόρτωση και καθαρισμός δεδομένων ελέγχου (Test Set)...")
label_names = ["sadness", "joy", "love", "anger", "fear", "surprise"]

# Φορτώνουμε το test set απευθείας από το Hugging Face
raw_test = load_dataset("dair-ai/emotion", split="test")
df_test = pd.DataFrame(raw_test)

# Καθαρίζουμε το test set από εσωτερικές διπλοεγγραφές
initial_len = len(df_test)
df_test_cleaned = df_test.drop_duplicates(subset=['text']).copy()
print(f"Test Set: Αφαιρέθηκαν {initial_len - len(df_test_cleaned)} διπλότυπα. Νέο μέγεθος: {len(df_test_cleaned)}")

# Το μετατρέπουμε πάλι σε αντικείμενο Hugging Face Dataset
clean_test_dataset = Dataset.from_pandas(df_test_cleaned)

# Επιλογή δειγμάτων με seed
test_samples = clean_test_dataset.shuffle(seed=69).select(range(1000))

# Ορισμός Τεχνικών Prompting
def get_prompt(text, choice):
    options = ", ".join(label_names)

    if choice == "1":  # Zero-Shot
        return (
            f"Classify the emotion in the following text.\n"
            f"You must respond with exactly one word from this list: [{options}]. Do not add any explanation.\n\n"
            f"Text: '{text}'\n"
            f"Emotion:"
        )

    elif choice == "2":  # Role-Prompting
        return (
            f"You are a specialized NLP sentiment analyst with deep expertise in fine-grained emotion classification.\n"
            f"Your task is to identify the single most dominant emotion expressed in the text.\n"
            f"Respond with exactly one word from this list: [{options}]. Output the label only — no explanation.\n\n"
            f"Text: '{text}'\n"
            f"Emotion label:"
        )

    elif choice == "3":  # Few-Shot
        return (
            f"Classify the emotion of the text. Respond with exactly one word from: [{options}].\n\n"
            f"Text: 'I feel like I am actually making a difference in this world'\n"
            f"Emotion: joy\n\n"
            f"Text: 'I feel so discouraged and like I am failing everyone'\n"
            f"Emotion: sadness\n\n"
            f"Text: 'I am so done with people treating me this way'\n"
            f"Emotion: anger\n\n"
            f"Text: 'I cannot stop thinking about how much I care for them'\n"
            f"Emotion: love\n\n"
            f"Text: 'I am feeling a bit apprehensive about the upcoming changes'\n"
            f"Emotion: fear\n\n"
            f"Text: 'I never thought this day would come, I am speechless'\n"
            f"Emotion: surprise\n\n"
            f"Text: 'I just feel empty and like nothing will ever get better'\n"
            f"Emotion: sadness\n\n"
            f"Text: '{text}'\n"
            f"Emotion:"
        )

    elif choice == "4":  # Chain of Thought
        return (
            f"Classify the emotion of the text. Choose strictly one label from: [{options}].\n\n"
            f"Follow these two steps:\n"
            f"Step 1: In one sentence, identify the key emotional words or phrases in the text.\n"
            f"Step 2: Output exactly 'Final Label: <emotion>' using one word from the list above.\n\n"
            f"Text: '{text}'\n"
            f"Step 1:"
        )

    elif choice == "5":  # Few-Shot Chain of Thought
        return (
            f"You are an expert NLP analyst. Classify the emotion of the text into exactly one of the following labels: [{options}].\n\n"
            f"Definitions:\n"
            f"- joy: Happiness, excitement, relief.\n"
            f"- sadness: Grief, feeling down, disappointment.\n"
            f"- anger: Frustration, irritation, outrage.\n"
            f"- fear: Anxiety, nervousness, terror.\n"
            f"- love: Affection, adoration, care.\n"
            f"- surprise: Astonishment, unexpected events.\n\n"
            f"Analyze the text by thinking step-by-step. First identify the trigger, then output the final label.\n\n"
            f"Text: 'I am feeling a bit apprehensive about the upcoming changes'\n"
            f"Analysis: The trigger is 'upcoming changes'. The word 'apprehensive' strongly indicates nervousness about the future. This aligns with fear.\n"
            f"Final Label: fear\n\n"
            f"Text: 'I feel such a strong affection for someone special in my life'\n"
            f"Analysis: The trigger is 'someone special'. 'Strong affection' indicates a deep personal bond, which aligns with love rather than just joy.\n"
            f"Final Label: love\n\n"
            f"Now analyze the following text:\n"
            f"Text: '{text}'\n"
            f"Analysis:"
        )

    elif choice == "6":  # Emotion Prompting
        return (
            f"I really need your help with this — it is important to me that you get it right.\n"
            f"Please read the text carefully and identify the emotion it expresses.\n"
            f"Respond with exactly one word from this list: [{options}]. No explanation needed.\n\n"
            f"Text: '{text}'\n"
            f"Emotion:"
        )

    return None

# Μενού και εκτέλεση
print("\n--- LLM Emotion Classification Tester ---")
print("1: Zero-Shot\n2: Role Prompting\n3: Few-Shot\n4: Chain-of-thought\n5: Few-Shot CoT\n6: Emotion Prompting")
choice = input("\nEnter choice (1,2,3,4,5,6): ").strip()

mode_name = {"1": "Zero-Shot", "2": "Role Prompting", "3": "Few-Shot", "4": "Chain-of-thought", "5": "Few-Shot CoT", "6": "Emotion Prompting"}.get(choice, "Unknown")

y_true = []
y_pred = []

print(f"\nRunning test using {mode_name}...")

# Ρυθμίσεις Batching
batch_size = 32
current_max_tokens = 60 if choice in ["4", "5"] else 5

# Loop (Με χρήση Batching)
for i in tqdm(range(0, len(test_samples), batch_size)):

    # Επιλογή των δειγμάτων για το τρέχον batch
    batch_items = test_samples.select(range(i, min(i + batch_size, len(test_samples))))

    prompts = [get_prompt(item['text'], choice) for item in batch_items]
    actual_labels = [label_names[item['label']] for item in batch_items]

    # Tokenization ολόκληρου του batch
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    input_length = inputs.input_ids.shape[1]

    # Παραγωγή κειμένου για όλο το batch ταυτόχρονα
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=current_max_tokens,
            temperature=0.01,
            pad_token_id=tokenizer.eos_token_id
        )

    # Αποκωδικοποίηση και επεξεργασία της κάθε απάντησης ξεχωριστά
    for j, output in enumerate(outputs):
        generated_tokens = output[input_length:]
        prediction_raw = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip().lower()

        # Καθαρισμός και Εξαγωγή Label
        if choice in ["4", "5"]:
            prediction = ""
            match = re.search(r'final label:\s*([a-z]+)', prediction_raw)
            if match:
                prediction = match.group(1).strip()
            else:
                found_labels = [label for label in label_names if label in prediction_raw]
                prediction = found_labels[-1] if found_labels else "unknown"
        else:
            words = re.findall(r'\w+', prediction_raw)
            prediction = words[0] if words else "unknown"

        y_true.append(actual_labels[j])
        y_pred.append(prediction)


# Τελικά Αποτελέσματα
print(f"\n{'='*40}")
print(f" FINAL RESULTS: {mode_name}")
print(f"{'='*40}")

# Υπολογισμός Accuracy
correct_predictions = sum(1 for true, pred in zip(y_true, y_pred) if true == pred)
accuracy = (correct_predictions / len(y_true)) * 100
print(f"Overall Accuracy: {accuracy:.2f}%\n")

# Εκτύπωση Classification Report
print("--- Classification Report ---")
print(classification_report(y_true, y_pred, labels=label_names, zero_division=0))

Φόρτωση και καθαρισμός δεδομένων ελέγχου (Test Set)...
Test Set: Αφαιρέθηκαν 0 διπλότυπα. Νέο μέγεθος: 2000

--- LLM Emotion Classification Tester ---
1: Zero-Shot
2: Role Prompting
3: Few-Shot
4: Chain-of-thought
5: Few-Shot CoT
6: Emotion Prompting

Enter choice (1,2,3,4,5,6): 4

Running test using Chain-of-thought...


  0%|          | 0/63 [00:00<?, ?it/s]Both `max_new_tokens` (=60) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, Fu


 FINAL RESULTS: Chain-of-thought
Overall Accuracy: 55.00%

--- Classification Report ---
              precision    recall  f1-score   support

     sadness       0.62      0.66      0.64       581
         joy       0.77      0.59      0.67       695
        love       0.28      0.36      0.31       159
       anger       0.52      0.51      0.52       275
        fear       0.58      0.44      0.50       224
    surprise       0.25      0.09      0.13        66

   micro avg       0.60      0.55      0.57      2000
   macro avg       0.50      0.44      0.46      2000
weighted avg       0.61      0.55      0.57      2000



In [ ]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!pip uninstall unsloth unsloth_zoo -y
!pip install unsloth unsloth_zoo --upgrade

Found existing installation: unsloth 2026.4.8
Uninstalling unsloth-2026.4.8:
  Successfully uninstalled unsloth-2026.4.8
Found existing installation: unsloth_zoo 2026.4.9
Uninstalling unsloth_zoo-2026.4.9:
  Successfully uninstalled unsloth_zoo-2026.4.9
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 4.6 MB/s eta 0:00:00
  Using cached unsloth_zoo-2026.4.9-py3-none-any.whl.metadata (32 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 8.0 MB/s eta 0:00:00
Using cached unsloth_zoo-2026.4.9-py3-none-any.whl (421 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 115.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 25